# Final-phase training on Colab GPU
**Owner:** Ishaan Karmokar · ICT-4442 ECG arrhythmia project

Runs every final-phase stage (grouped cross-validation on DS1, ablations, the CNN's DE/PSO search, final 3-seed training) on a GPU.

**How to use:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Allow Google Drive access when asked.

Every finished run is saved to `MyDrive/ecg-final/` straight away and skipped when a cell is re-run, so if Colab disconnects, just reconnect and **Run all** again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU - set Runtime > Change runtime type > T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ecg-final'
import os
assert os.path.exists(f'{DRIVE}/ecg_bundle.zip'), 'ecg_bundle.zip not found in MyDrive/ecg-final'

## Setup: code, packages, data

In [ ]:
%%bash
set -e
DRIVE=/content/drive/MyDrive/ecg-final
rm -rf /content/ecg-arrhythmia && mkdir -p /content/ecg-arrhythmia
unzip -q -o $DRIVE/ecg_bundle.zip -d /content/ecg-arrhythmia
cd /content/ecg-arrhythmia
# results and weights live on Drive, so they survive disconnects
mkdir -p $DRIVE/results_final $DRIVE/saved_models_final results saved_models logs
rm -rf results/final saved_models/final
ln -s $DRIVE/results_final results/final
ln -s $DRIVE/saved_models_final saved_models/final
pip -q install wfdb PyWavelets imbalanced-learn
python -c "import tensorflow as tf; print('TF', tf.__version__, tf.config.list_physical_devices('GPU'))"

In [ ]:
%%bash
set -e
DRIVE=/content/drive/MyDrive/ecg-final
cd /content/ecg-arrhythmia
mkdir -p data/processed
if [ -f $DRIVE/data/beats.npz ] && [ -f $DRIVE/data/context.npz ]; then
  cp $DRIVE/data/*.npz data/processed/ && echo 'dataset restored from Drive'
else
  python src/download_data.py && python src/build_dataset.py | tail -9
  mkdir -p $DRIVE/data && cp data/processed/*.npz $DRIVE/data/
fi
md5sum data/processed/beats.npz

## Stage A: Random Forest and CNN ablations (grouped CV on DS1)

In [ ]:
%cd /content/ecg-arrhythmia
!python -u src/models/m1_random_forest.py --mode cv 2>&1 | grep -v NodeDef | tee -a logs/stageA.log
!python -u src/models/m2_cnn1d.py --mode cv-ablation 2>&1 | grep -v NodeDef | tee -a logs/stageA.log

## Stage B: CNN hyperparameter search with DE and PSO (objective = grouped-CV macro-F1)

In [ ]:
%cd /content/ecg-arrhythmia
!python -u src/models/m2_cnn1d.py --mode cv-search --opt de  2>&1 | grep -v NodeDef | tee -a logs/stageB.log
!python -u src/models/m2_cnn1d.py --mode cv-search --opt pso 2>&1 | grep -v NodeDef | tee -a logs/stageB.log
!python -u src/models/m2_cnn1d.py --mode cv-select           2>&1 | grep -v NodeDef | tee -a logs/stageB.log

## Stage C: BiGRU ablations

In [ ]:
%cd /content/ecg-arrhythmia
!python -u src/models/m3_bigru.py --mode cv 2>&1 | grep -v NodeDef | tee -a logs/stageC.log

## Stage D: Transformer ablations

In [ ]:
%cd /content/ecg-arrhythmia
!python -u src/models/m4_transformer.py --mode cv 2>&1 | grep -v NodeDef | tee -a logs/stageD.log

## Stage E: final models on all of DS1, 3 seeds, DS2 scored once

In [ ]:
%cd /content/ecg-arrhythmia
!python -u src/models/m1_random_forest.py --mode final 2>&1 | grep -v NodeDef | tee -a logs/stageE.log
!python -u src/models/m2_cnn1d.py --mode cv-final       2>&1 | grep -v NodeDef | tee -a logs/stageE.log
!python -u src/models/m3_bigru.py --mode final          2>&1 | grep -v NodeDef | tee -a logs/stageE.log
!python -u src/models/m4_transformer.py --mode final    2>&1 | grep -v NodeDef | tee -a logs/stageE.log

## Done: summary (everything is already saved in MyDrive/ecg-final)

In [ ]:
import json, glob, os
os.system(f'cp logs/*.log {DRIVE}/ 2>/dev/null')
for p in sorted(glob.glob('results/final/*.json')):
    r = json.load(open(p))
    print(os.path.basename(p), {k: round(v['macro_f1']['mean'], 4) for k, v in r.items()})
open(f'{DRIVE}/ALL_STAGES_DONE.txt', 'w').write('done')